<a href="https://colab.research.google.com/github/aggarwal-navya/DeepLearning/blob/main/RNN_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [132]:
import pandas as pd

In [133]:
df= pd.read_csv("/content/IMDB Dataset.csv")

In [134]:
df.shape

(50000, 2)

In [135]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [136]:
df.isnull().sum()

,0
review,0
sentiment,0


In [137]:
df.drop_duplicates(inplace=True)

In [138]:
df.shape

(49582, 2)

In [139]:
#pre-processing

In [140]:
### convert to lower case

In [141]:
df["review"]= df["review"].str.lower()

In [142]:
## removing url

In [143]:
import re
  #sample_text= "abc is the word, abc" # abc => xyz
  #new_text= re.sub("abc", "xyz", sample_text)

In [144]:
def remove_url(text):
  text= re.sub(r"http\S+" , "", text)
  return text

df["review"] = df["review"].apply(remove_url)

In [145]:
## remove punctuation

In [146]:
def remove_punc(text):
  text= re.sub(r"[^A-Za-z0-9\s]" , "", text)
  return text

df["review"] = df["review"].apply(remove_punc)

In [147]:
def remove_html(text):
  text= re.sub(r"<.*?>" , "", text)
  return text

df["review"] = df["review"].apply(remove_html)

In [148]:
## removing stopwords

In [149]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [150]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [151]:
#sample_text= "i like python"
#tokens= word_tokenize(sample_text)

In [152]:
def remove_stopwords(text):
  tokens= word_tokenize(text)
  stop_words= stopwords.words("english")

  for words in stop_words:
    if words in stop_words:
      text= text.replace(words, "")
  return text

df["review"] = df["review"].apply(remove_stopwords)

In [153]:
df.head()

,review,sentiment
0,ne f r r h enne h fer wchng 1 z epe u hke ...,positive
1,wnerful lle prucn br br flng echnque r unun...,positive
2,hugh w wnerful w pen e n h uer eken ng n...,positive
3,bc fl w lle b jke hnk zbe n cle pn fg...,negative
4,peer e l n e f ne vu unnng fl wch r e ffer...,positive


In [154]:
### stemming
## running -> run
# played -> play
# portersteaming

In [155]:
from nltk.stem import PorterStemmer

In [156]:
def stemming(text):
  pt= PorterStemmer()
  stemmed_words=[]

  tokens= word_tokenize(text)
  for token in tokens:
    stemmed_token= pt.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)
df["review"]= df["review"].apply(stemming)

In [157]:
df.head()

,review,sentiment
0,ne f r r h enn h fer wchng 1 z epe u hke rgh e...,positive
1,wner lle prucn br br flng echnqu r unung r leb...,positive
2,hugh w wner w pen e n h uer eken ng n r cnne e...,positive
3,bc fl w lle b jke hnk zbe n cle pn fghng ebr b...,negative
4,peer e l n e f ne vu unnng fl wch r e ffer u v...,positive


In [158]:
# encoding

In [159]:
from sklearn.preprocessing import LabelEncoder
le= LabelEncoder()

df["sentiment"]= le.fit_transform(df["sentiment"])

In [160]:
y= df["sentiment"]

In [161]:
## vectorization

In [162]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf= TfidfVectorizer(max_features=5000)
X= tf.fit_transform(df["review"])

In [163]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3523151 stored elements and shape (49582, 5000)>

In [164]:
# Dataset and Dataloaders

In [165]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=42)

In [166]:
X_train.shape

(39665, 5000)

In [167]:
X_test.shape

(9917, 5000)

In [168]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [169]:
X_train= X_train.toarray()
X_test= X_test.toarray()

In [170]:
train_set= TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set= TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [171]:
train_loader= DataLoader(train_set, batch_size=32, shuffle=True)
test_loader= DataLoader(test_set, batch_size=32, shuffle=True)

In [172]:
# buildig RNN

In [173]:
import torch.nn as nn
import torch.optim as optim

In [180]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [181]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [182]:
# training RNN

In [185]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.6736904978752136
epoch = 2/10 and loss = 0.11770930141210556
epoch = 3/10 and loss = 0.5129870176315308
epoch = 4/10 and loss = 0.4498128890991211
epoch = 5/10 and loss = 0.4436306953430176
epoch = 6/10 and loss = 0.11934076994657516
epoch = 7/10 and loss = 0.41310858726501465
epoch = 8/10 and loss = 0.2090912014245987
epoch = 9/10 and loss = 0.3830508887767792
epoch = 10/10 and loss = 0.34604611992836


In [186]:
# evaluation

model.eval()

with torch.no_grad():
  correct_vals= 0
  tot_vals = 0

  for Xb, yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5). float()

    tot_vals += yb.size(0)
    correct_vals += (predicted == yb). sum().item()

  print(f"Accuracy = {correct_vals/tot_vals*100}%")

Accuracy = 83.35182010688716%
